### 테이블 나누기
- 대출번호/회원번호/회원전화/책제목/책저자/대출일
- 1/이현근/010-111/판다스입문/다니엘/2026-08-01

- 위처럼 저장했을 때의 문제
    - 책 빌릴 때 마다 -> 회원이름 /전화번호 데이터 조회
    - 전화번호 변경 시 문제 -> 모든 행를 전부 조작해야 한다
- 테이블 나누기 필요

- 회원 테이블 : 회원정보는 한번만 기록
- 책 테이블 : 책정보도 한번만
- 대출 테이블 : 누가(회원정보) 어떤 책을(책번호) -> 연결 정보만 저장

####  세개의 테이블 관계 - ERD로 보기
- 1:N 관계 - 회원 1명은 대출을 여러 건 진행할 수 있음
- foreign key : 이 열은 저쪽 테이블 기본키를 참조 명시
- 

In [2]:
import sqlite3
import pandas as pd

In [3]:
import sqlite3

conn = sqlite3.connect("library_test.db")
cur = conn.cursor()


In [4]:
conn = sqlite3.connect("library_test.db")
conn.execute("PRAGMA foreign_keys = ON")

In [5]:
# 1번째 테이블 만들기 - 회원정보
cur.execute("""
CREATE TABLE members (
    member_id INTEGER PRIMARY KEY,
    name TEXT,
    phone TEXT
)
""")
conn.commit()

OperationalError: table members already exists

In [6]:
# 2번째 테이블 만들기 - books
cur.execute("""
CREATE TABLE books (
    book_id INTEGER PRIMARY KEY,
    title TEXT,
    author TEXT
)
""")
conn.commit()

OperationalError: table books already exists

In [7]:
# 3번째 테이블 만들기 - 연결정보
cur.execute("""
CREATE TABLE loans (
    loan_id INTEGER PRIMARY KEY,
    member_id INTEGER,
    book_id INTEGER,
    loan_date TEXT,
    FOREIGN KEY (member_id) REFERENCES members(member_id),
    FOREIGN KEY (book_id) REFERENCES books(book_id)
)
""")
conn.commit()

OperationalError: table loans already exists

In [8]:
# 책정보 등록해보기
member_list = [
    (1, "이현근", "010-1111"),
    (2, "김수연", "010-2222"),
    (3, "송수림", "010-3333"),
    (4, "주승우", "010-4444"),
]
cur.executemany("INSERT INTO members VALUES (?, ?, ?)", member_list)
conn.commit()

In [9]:
# 책정보 등록해보기
book_list = [
    (101, "오늘만 사는법", "이현근2"),
    (102, "추석에 맛있는거 먹는법", "김수연3"),
    (103, "신기한 여행", "에드워드"),
    (104, "핸즈온 머신러닝", "오렐리앙"),
]
cur.executemany("INSERT INTO books VALUES (?, ?, ?)", member_list)
conn.commit()

In [10]:
# 대출 기록 넣기 - 회원번호-책번호로만 연결
loan_list = [
    (1, 1, 101, "2026-08-01"),
    (2, 3, 104, "2026-08-02"),
    (3, 1, 102, "2026-08-03"),
    (4, 2, 103, "2026-08-04"),
    (5, 4, 102, "2026-08-05"),
    (6, 4, 104, "2026-08-08"),
]
cur.executemany("INSERT INTO loans VALUES (?, ?, ?, ?)", loan_list)


In [11]:
conn.commit()

join

In [12]:
data = pd.read_sql("""
SELECT loans.loan_date AS 대출일,
       members.name    AS 회원, 
       books.title      AS 책제목
FROM loans
JOIN members ON loans.member_id = members.member_id
JOIN books   ON loans.book_id   = books.book_id
""", conn)
data

,대출일,회원,책제목


In [13]:
# 대출 건수가 많은 사람 조회하기
# groupby 회원별 대출수
data = pd.read_sql("""
SELECT members.name     AS 회원, 
       COUNT(*)         AS 대출권수
FROM loans
JOIN members ON loans.member_id = members.member_id
GROUP BY members.name
""", conn)
data

,회원,대출권수
